In [2]:
#| default_exp multihop

In [3]:
#| export
import torch, numpy as np, json, os, torch.nn.functional as F

from tqdm.auto import tqdm
from typing import Dict, List, Optional
from fastcore.utils import patch

from transformers import AutoTokenizer
from transformers.modeling_utils import unwrap_model
from transformers.trainer_pt_utils import nested_concat

from xcai.learner import XCLearner, XCPredictionOutput

In [4]:
#| export
class MultihopBeamSearchScorer:
    """
    Vectorized beam search scorer specifically optimized for dense multihop document retrieval.
    """
    def __init__(self, batch_size: int, num_beams: int, device: torch.device):
        self.batch_size, self.num_beams, self.device = batch_size, num_beams, device
        self.beam_scores = torch.zeros((batch_size, num_beams), dtype=torch.float, device=device)
        self.beam_scores[:, 1:] = -1e9

    def process(self, next_scores: torch.Tensor, next_docs: torch.Tensor, beam_paths: torch.Tensor):
        batch_size, current_beam_size, current_topk = next_scores.shape
        
        active_beam_scores = self.beam_scores[:, :current_beam_size]
        next_scores = next_scores + active_beam_scores.unsqueeze(-1)
        
        next_scores = next_scores.view(batch_size, -1)
        k = min(self.num_beams, next_scores.shape[1])
        topk_scores, topk_indices = torch.topk(next_scores, k, dim=-1)

        self.beam_scores[:, :k] = topk_scores
        beam_indices = torch.div(topk_indices, current_topk, rounding_mode="floor")

        next_docs_flat = next_docs.contiguous().view(batch_size, -1)
        selected_docs = torch.gather(next_docs_flat, 1, topk_indices)

        if beam_paths.shape[-1] == 0:
            new_paths = selected_docs.unsqueeze(-1)
        else:
            gathered_paths = torch.gather(
                beam_paths, 1, beam_indices.unsqueeze(-1).expand(-1, -1, beam_paths.shape[-1])
            )
            new_paths = torch.cat([gathered_paths, selected_docs.unsqueeze(-1)], dim=-1)
            
        return topk_scores, selected_docs, new_paths, beam_indices
            
    def finalize(self, beam_paths, topk_scores):
        return beam_paths.cpu(), topk_scores.cpu()


In [7]:
#| export
class MultihopLearner(XCLearner):

    def __init__(
        self,
        tokenizer,
        **kwargs
    ):
        super().__init__(**kwargs)
        
        if isinstance(tokenizer, str):
            assert os.path.exists(tokenizer), "Configuration file not found."
            with open(tokenizer) as file:
                config = json.load(file)
                mname = config[list(config)[0]]["parameters"]["tokenizer"]
            self.processing_class = AutoTokenizer.from_pretrained(mname)
        else:
            self.processing_class = tokenizer


    def evaluation_loop(
        self,
        dataset,
        beam_size: Optional[int] = 5,
        num_hops: Optional[int] = 3,
        topk_per_hop: Optional[int] = 10,
    ):
        """
        Efficient vectorized multi-hop beam search retrieval.
        """
        # Index creation
        if self._perform_representation(unwrap_model(self.model)):
            self._build_lbl_index(dataset)
            
        # Label text extraction

        lbl_info = self.eval_dataset.lbl_info if dataset.lbl_info is None else dataset.lbl_info
        lbl_info = self.train_dataset.lbl_info if lbl_info is None else lbl_info

        assert lbl_info is not None, "`lbl_info` cannot be empty"
        label_text = lbl_info["input_text"]

        # Multihop setup
        dataloader = self.get_test_dataloader(dataset)

        model = self._wrap_model(self.model, training=False, dataloader=dataloader)
        device = "cpu" if self.args.use_cpu_for_searching else model.device
        
        if len(self.accelerator._models) == 0 and model is self.model:
            model = self.accelerator.prepare_model(model, evaluation_mode=True)
            
        model.eval()

        beam_output = dict()
        all_beam_output = {i:dict() for i in range(num_hops)}

        for step, inputs in tqdm(enumerate(dataloader), total=len(dataloader)):
            assert "data_input_text" in inputs, "`data_input_text` not present in the inputs"

            base_queries = inputs["data_input_text"]
            batch_size = len(base_queries)

            labels = (
                {'targ_idx':inputs[self.args.target_indices_key], 'targ_ptr':inputs[self.args.target_pointer_key]}
                if self.args.target_indices_key in inputs else None
            )
            if labels is not None and self.args.target_scores_key in inputs:
                labels.update({'targ_score':inputs[self.args.target_scores_key]})
                
            for k, v in labels.items():
                beam_output.setdefault(k, []).append(v)

            scorer = MultihopBeamSearchScorer(batch_size, beam_size, device)

            beam_paths = torch.zeros((batch_size, beam_size, 0), dtype=torch.long, device=device)
            current_queries, current_inputs = [[q] for q in base_queries], inputs

            for hop in range(num_hops):
                loss, output = self.prediction_step(
                    model,
                    current_inputs,
                    prediction_loss_only=False,
                    predict_with_generation=False,
                    predict_with_representation=True,
                )
                
                pred_idx, pred_score, pred_ptr = output["pred_idx"], output["pred_score"], output["pred_ptr"]
                
                topk = pred_ptr[0]
                pred_idx, pred_score = pred_idx.view(batch_size, -1, topk), pred_score.view(batch_size, -1, topk)
                pred_score = F.log_softmax(pred_score, dim=-1)

                # Mask already predicted labels

                if hop > 0:
                    mask = (pred_idx.unsqueeze(-1) == beam_paths.unsqueeze(2))
                    mask = mask.any(dim=-1)
                    pred_score = pred_score.masked_fill(mask, -1e9)

                    pred_score, sorted_indices = torch.sort(pred_score, dim=-1, descending=True)
                    pred_idx = torch.gather(pred_idx, dim=-1, index=sorted_indices)

                current_topk = min(topk, topk_per_hop)
                pred_idx, pred_score = pred_idx[:, :, :current_topk], pred_score[:, :, :current_topk]

                # Perform beam-search
                
                topk_scores, selected_docs, beam_paths, beam_indices = scorer.process(
                    pred_score, pred_idx, beam_paths,
                )

                paths, scores = scorer.finalize(beam_paths, topk_scores)
                all_beam_output[hop].setdefault("paths", []).append(paths)
                all_beam_output[hop].setdefault("scores", []).append(scores)

                # Reformulated query

                k = topk_scores.shape[1]
                
                if hop < num_hops - 1:
                    new_queries = []
                    
                    for b in range(batch_size):
                        batch_new_queries = []

                        for h_idx in range(k):
                            prev_beam = beam_indices[b, h_idx].item()
                            doc_id = selected_docs[b, h_idx].item()
                            prev_query = current_queries[b][prev_beam]

                            expanded = prev_query + " " + self.processing_class.sep_token + " " + label_text[doc_id]
                            batch_new_queries.append(expanded)

                        new_queries.append(batch_new_queries)

                    current_queries = new_queries
                    flat_queries = [q for batch_q in current_queries for q in batch_q]

                    tokenized = self.processing_class(
                        flat_queries,
                        padding=True,
                        truncation=True,
                        max_length=self.args.max_length if hasattr(self.args, "max_length") else 512,
                        return_tensors="pt"
                    )
                    tokenized = {k:v.to(model.device) for k,v in tokenized.items()}

                    current_inputs = {
                        "data_input_ids": tokenized["input_ids"],
                        "data_attention_mask": tokenized["attention_mask"],
                    }
                    if "token_type_ids" in tokenized:
                        current_inputs["data_token_type_ids"] = tokenized["token_type_ids"]

            final_paths, final_scores = scorer.finalize(beam_paths, topk_scores)
            
            beam_output.setdefault("paths", []).append(final_paths)
            beam_output.setdefault("scores", []).append(final_scores)

        # Collect outputs
        for k,v in beam_output.items():
            output[k] = torch.cat(v, dim=0).to("cpu") if len(v) else None
            
        all_output = dict()
        for hop in range(num_hops):
            all_output[hop] = dict()
            for k,v in all_beam_output[hop].items():
                all_output[hop][k] = torch.cat(v, dim=0).to("cpu") if len(v) else None

        return output, all_output
        

    def predict(
        self,
        test_dataset,
        ignore_keys: Optional[List[str]] = None,
        metric_key_prefix: str = "test",
        **kwargs
    ):
        output, all_output = self.evaluation_loop(test_dataset, **kwargs)
        
        # debug
        
        metric_input_keys = ["targ_idx", "targ_ptr", "targ_score"]
        for hop, o in all_output.items():
            print(f"Hop number: {hop}")

            paths, scores = o.get("paths"), o.get("scores")

            inputs = {k:output[k] for k in metric_input_keys if k != "targ_score" or k in output}

            batch_size, num_items = paths.shape[0], paths.shape[1] * paths.shape[2]
            
            # scores = scores.unsqueeze(-1).expand(-1, -1, paths.shape[2])
            scores = torch.arange(num_items, 0, -1).unsqueeze(0).expand(batch_size, -1)
            
            inputs["pred_score"] = scores.flatten()
            inputs["pred_idx"] = paths.flatten()
            inputs["pred_ptr"] = torch.full((batch_size,), num_items)

            metrics = self.compute_metrics(**inputs)
            
            print(metrics)

        # debug

        # paths, scores = output.get("paths"), output.get("scores")

        # pred_ptr = torch.full((scores.shape[0],), scores.shape[1])
        # pred_score = scores.flatten()
        # pred_idx = paths[:, :, -1].flatten()
        #
        # metrics, metric_input_keys = {}, ["targ_idx", "targ_ptr", "targ_score"]
        # if (
        #     self.compute_metrics is not None and paths is not None and
        #     "targ_idx" in output and output["targ_idx"] is not None
        # ):
        #     num_hops = paths.shape[-1]
        #     inputs = {o:output[o] for o in metric_input_keys if o != "targ_score" or o in output}
        #     inputs["pred_ptr"] = pred_ptr
        #     inputs["pred_score"] = pred_score

        #     for hop in range(num_hops):
        #         inputs["pred_idx"] = paths[:, :, hop].flatten()
        #         hop_metrics = self.compute_metrics(**inputs)
        #         for k, v in hop_metrics.items():
        #             metrics[f"{metric_key_prefix}_hop_{hop}_{k}"] = v

        # return XCPredictionOutput(pred_idx=pred_idx, pred_score=pred_score, pred_ptr=pred_ptr, metrics=metrics)
        
        return XCPredictionOutput(pred_idx=inputs["pred_idx"], pred_score=inputs["pred_score"], pred_ptr=inputs["pred_ptr"], metrics=metrics)

    
    def evaluate(
        self,
        eval_dataset = None,
        ignore_keys: Optional[List[str]] = None,
        metric_key_prefix: str = "eval",
        **kwargs
    ):
        eval_dataset = eval_dataset if eval_dataset is not None else self.eval_dataset
        output = self.predict(
            test_dataset=eval_dataset,
            ignore_keys=ignore_keys,
            metric_key_prefix=metric_key_prefix,
            **kwargs
        )

        self.log(output.metrics)
        self.control = self.callback_handler.on_evaluate(self.args, self.state, self.control, output.metrics)
        return output.metrics
